In [8]:
import pandas as pd
import os

In [9]:
# 阶段 1：读取并查看整体结构

# 读取融合后的元数据（不做类型强制，避免误判）
df = pd.read_csv("../data/metadata_merged_raw.csv", low_memory=False)

# 1) 行列规模：了解数据量级
print("shape (rows, cols):", df.shape)

# 2) 列名清单：为后续选择主键/文本列/数值列做准备
print("columns:", df.columns.tolist())

# 3) 前几行示例：快速感知字段取值和是否有意外空值/格式
print(df.head(3))


shape (rows, cols): (521735, 17)
columns: ['Id', 'CreatorUserId', 'OwnerUserId', 'OwnerOrganizationId', 'CurrentDatasetVersionId', 'CurrentDatasourceVersionId', 'ForumId', 'Type', 'CreationDate', 'LastActivityDate', 'TotalViews', 'TotalDownloads', 'TotalVotes', 'TotalKernels', 'Medal', 'MedalAwardDate', 'Tags']
       Id  CreatorUserId  OwnerUserId  OwnerOrganizationId  \
0  402034        2188609    2188609.0                  NaN   
1  402031        3792299    3792299.0                  NaN   
2   39875              1    1891177.0                  NaN   

   CurrentDatasetVersionId  CurrentDatasourceVersionId  ForumId     Type  \
0                 771250.0                    792478.0   414197  Dataset   
1                 771247.0                    792475.0   414194  Dataset   
2                 285978.0                    298457.0    48350  Dataset   

          CreationDate LastActivityDate  TotalViews  TotalDownloads  \
0  10/31/2019 01:07:32       10/31/2019        2706           

In [10]:
# 阶段 2：类型与缺失概览 + 主键检查
# 1) 查看各列数据类型（推断是否需要后续类型纠正，如日期/整数被读成对象或浮点）
print(df.dtypes)

# 2) 计算每列的缺失比例（按从高到低排序，先关注缺失多的列）
na_rate = df.isna().mean().sort_values(ascending=False).round(4)
print(na_rate.head(10))  # 先看缺失最多的前10列

# 3) 主键健全性：检查 Id 是否唯一且无缺失（一数据集一行的前提）
print("rows:", len(df),
      "unique_Id:", df["Id"].nunique(),
      "null_Id:", df["Id"].isna().sum())

# 4) 粗看“热度/参与度”数值列的统计分布（是否有极端长尾、负值异常等）
print(df[["TotalViews","TotalDownloads","TotalVotes","TotalKernels"]].describe())


Id                              int64
CreatorUserId                   int64
OwnerUserId                   float64
OwnerOrganizationId           float64
CurrentDatasetVersionId       float64
CurrentDatasourceVersionId    float64
ForumId                         int64
Type                           object
CreationDate                   object
LastActivityDate               object
TotalViews                      int64
TotalDownloads                  int64
TotalVotes                      int64
TotalKernels                    int64
Medal                         float64
MedalAwardDate                 object
Tags                           object
dtype: object
OwnerOrganizationId           0.9951
Medal                         0.9594
MedalAwardDate                0.9440
Tags                          0.5887
OwnerUserId                   0.0049
CurrentDatasourceVersionId    0.0005
CurrentDatasetVersionId       0.0005
TotalDownloads                0.0000
TotalKernels                  0.0000
TotalVo

In [11]:
# 阶段 3a：将字符串日期解析为 datetime，便于后续计算“年龄/新鲜度”
df["CreationDate_dt"] = pd.to_datetime(df["CreationDate"], errors="coerce")
df["LastActivityDate_dt"] = pd.to_datetime(df["LastActivityDate"], errors="coerce")

# 快速检查解析效果与缺失率
print(df[["CreationDate","CreationDate_dt","LastActivityDate","LastActivityDate_dt"]].head(3))
print("NA rate (CreationDate_dt, LastActivityDate_dt):",
      df["CreationDate_dt"].isna().mean().round(4),
      df["LastActivityDate_dt"].isna().mean().round(4))


          CreationDate     CreationDate_dt LastActivityDate  \
0  10/31/2019 01:07:32 2019-10-31 01:07:32       10/31/2019   
1  10/31/2019 00:58:15 2019-10-31 00:58:15       10/31/2019   
2  08/01/2018 00:10:55 2018-08-01 00:10:55       08/01/2018   

  LastActivityDate_dt  
0          2019-10-31  
1          2019-10-31  
2          2018-08-01  
NA rate (CreationDate_dt, LastActivityDate_dt): 0.0 0.0


In [12]:
# 阶段 3b：派生时间特征（年龄/最近活跃天数/是否近30天活跃）
# 目的：
# - 以“今天”为参照，计算数据集的创建“年龄（天）”与“距离最近活跃的天数”
# - 给出一个布尔特征：近30天是否活跃，用于后续过滤/排序

import numpy as np
today = pd.Timestamp.today().normalize()                               # 取今天（去掉时分秒）
df["age_days"] = (today - df["CreationDate_dt"]).dt.days               # 从创建日到今天的天数
df["days_since_last_activity"] = (today - df["LastActivityDate_dt"]).dt.days  # 距最近活跃的天数
df["active_30d"] = df["days_since_last_activity"].le(30).astype("boolean")    # 是否近30天内活跃

# 简要统计，确认派生是否正常
print(df[["age_days","days_since_last_activity","active_30d"]].describe(include="all"))


             age_days  days_since_last_activity active_30d
count   521735.000000             521735.000000     521735
unique            NaN                       NaN          2
top               NaN                       NaN      False
freq              NaN                       NaN     509785
mean       829.033172                828.763644        NaN
std        659.518643                655.490268        NaN
min          0.000000                  1.000000        NaN
25%        289.000000                290.000000        NaN
50%        650.000000                651.000000        NaN
75%       1274.000000               1275.000000        NaN
max       3684.000000               2945.000000        NaN


In [13]:
# 阶段 4b｜类型纠正（浮点→可空整型）+ 标签缺失填充与标记
# 目的：
# - 将因缺失而被读成 float 的“整数列”改为可空整型 Int64，避免小数点与类型歧义
# - 将缺失的 Tags 填为空串，并增加 has_tags 便于筛选

cols_to_int = ["OwnerUserId", "OwnerOrganizationId", "CurrentDatasetVersionId", "CurrentDatasourceVersionId", "Medal"]
for c in cols_to_int:
    if c in df.columns:                                # 仅当列存在时才转换，避免 KeyError
        df[c] = df[c].astype("Int64")                  # 转为可空整型（支持 NA）

df["Tags"] = df["Tags"].fillna("")                     # 缺失标签填为空串，便于字符串操作
df["has_tags"] = df["Tags"].str.len().gt(0)            # 是否包含至少一个标签（布尔）

print(df[cols_to_int].dtypes)                          # 快速检查被转换列的类型
print(df[["Id","Tags","has_tags"]].head(3))            # 预览结果是否符合预期


OwnerUserId                   Int64
OwnerOrganizationId           Int64
CurrentDatasetVersionId       Int64
CurrentDatasourceVersionId    Int64
Medal                         Int64
dtype: object
       Id               Tags  has_tags
0  402034                        False
1  402031  health conditions      True
2   39875           bigquery      True


In [ ]:
# 阶段 5c｜保存清洗后的结果到 CSV（仅此一步）
# - 设置输出路径
# - 保存为 CSV（不带索引）
# - 打印确认信息（路径与形状）

out_path = ".." \
"/data/metadata_merged.csv"     # 指定输出文件名与位置
df.to_csv(out_path, index=False)            # 保存当前清洗后的 DataFrame
print("saved:", out_path, "shape:", df.shape)  # 确认保存成功与行列规模


saved: ../data/metadata_merged.csv shape: (521735, 23)
